# BLOQUE 01 – Transporte público
## Objetivo
Construir capa consolidada demográfica normalizada por unidad espacial.
## Inputs
- transporte.shp
- masivo.shp
## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output
1_D4C1_Movilidad.gpkg
- 1_D4C1V2_transporte_público.shp


In [2]:
import geopandas as gpd
import pandas as pd
import os

In [83]:
CRS_MASTER = "MEXICO_ITRF_2008_LCC"

In [91]:
# Rutas alex
ruta_transporte = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\Rutas de TP\Rutas de Transporte.gpkg"
ruta_masivo = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\TP Masivo\TP Masivo (rutas).gpkg"

ruta_p_tp = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\Rutas de TP\TRANSPORTE PUBLICO SITRAN (rodo - GTFD 2024)\PARADAS.shp"
ruta_p_masivo = r"C:\Users\alflo\Desktop\DAL_LAB\data_raw\TP Masivo\P TPM (con L4).gpkg"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D4C1_Movilidad.gpkg"

# # Rutas aldo
# ruta_transporte = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\Rutas de TP\Rutas de Transporte.gpkg"
# ruta_masivo = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\data_raw\TP Masivo\TP Masivo.gpkg"
# ruta_salida = r"C:\Users\aldov\OneDrive\Escritorio\DAL_LABORATY\outputs\1_D4C1_Movilidad.gpkg"

In [92]:
# --- 2. PROCESAMIENTO DE RUTAS DE TRANSPORTE (COLECTIVO) ---
gdf_colectivo = gpd.read_file(ruta_transporte)
gdf_colectivo['Categoría'] = 'colectivo'

gdf_colectivo = gdf_colectivo[gdf_colectivo['Situacion_'] == "Operando"].copy()

# Borrar columnas específicas de Colectivo
cols_borrar_colectivo = ["Nombre_del","Nombre_de_", "Nombre_de0","Clasificac", 'Situacion_']
gdf_colectivo = gdf_colectivo.drop(columns=cols_borrar_colectivo, errors='ignore')

#renombrar categoría 
gdf_colectivo.rename(columns={'tipo_de_ru': 'Tipo'}, inplace=True)

In [93]:
# --- 3. PROCESAMIENTO DE TP MASIVO (MASIVO) ---
gdf_masivo = gpd.read_file(ruta_masivo)
gdf_masivo['Categoría'] = 'masivo'

gdf_masivo = gdf_masivo[gdf_masivo['estado_act'] == "Existente"].copy()

# Borrar columnas específicas de Masivo
cols_borrar_masivo = ["nombre_rut", "estado_act"]
gdf_masivo = gdf_masivo.drop(columns=cols_borrar_masivo, errors='ignore')

#renombrar categoría 
gdf_masivo.rename(columns={'tipo_siste': 'Tipo'}, inplace=True)

In [94]:
# --- 4. HOMOLOGAR COORDENADAS (CRS) ---
gdf_masivo = gdf_masivo.to_crs(CRS_MASTER)
gdf_colectivo = gdf_colectivo.to_crs(CRS_MASTER)

# --- 5. UNIÓN Y GUARDADO ---
gdf_final = pd.concat([gdf_colectivo, gdf_masivo], ignore_index=True)
gdf_final = gpd.GeoDataFrame(gdf_final, geometry='geometry', crs=gdf_colectivo.crs)

In [95]:
# Guardar en la ruta especificada
gdf_final.to_file(ruta_salida, layer='1_D4C1V2_transporte_público', driver="GPKG")

# Paradas

In [84]:
paradas_masivo = gpd.read_file(ruta_p_masivo)

paradas_masivo['Categoría'] = 'masivo'

paradas_masivo = paradas_masivo[paradas_masivo['Estado'] == "Existente"].copy()
                    
# Borrar columnas específicas de Colectivo
cols_borrar_p_masivo = ['NORM','PONDERACIO','POND','Nombre_de_','Estado']
paradas_masivo = paradas_masivo.drop(columns=cols_borrar_p_masivo, errors='ignore')

#renombrar categoría 
paradas_masivo.rename(columns={'L�nea_y_si': 'Linea'}, inplace=True)
paradas_masivo.rename(columns={'Sistema': 'Tipo'}, inplace=True)

In [85]:
paradas_tp = gpd.read_file(ruta_p_tp)

paradas_tp = paradas_tp[paradas_tp['TIPO_TP'] == 'Convencion'].copy()

paradas_tp = paradas_tp.drop_duplicates(subset=["stop_lat", "stop_lon"]).copy()

paradas_tp = paradas_tp.reset_index(drop=True)

# Borrar columnas específicas de Colectivo
cols_borrar_p_tp = ['stop_name','stop_lat','stop_lon','PESO','NORM','TIPO_TP','PONDERADO']
paradas_tp = paradas_tp.drop(columns=cols_borrar_p_tp, errors='ignore')

paradas_tp['Categoría'] = 'colectivo'
paradas_tp['Tipo'] = 'Bus'

#renombrar categoría 
paradas_tp.rename(columns={'stop_id': 'Linea'}, inplace=True)

In [86]:
# --- 4. HOMOLOGAR COORDENADAS (CRS) ---
paradas_masivo = paradas_masivo.to_crs(CRS_MASTER)
paradas_tp = paradas_tp.to_crs(CRS_MASTER)

# --- 5. UNIÓN Y GUARDADO ---
gdf_final = pd.concat([paradas_tp, paradas_masivo], ignore_index=True)
gdf_final = gpd.GeoDataFrame(gdf_final, geometry='geometry', crs=paradas_tp.crs)

In [87]:
# Guardar en la ruta especificada
gdf_final.to_file(ruta_salida, layer='1_D4C1V2_transporte_público_paradas', driver="GPKG")